## What is MCP?

**Model Context Protocol (MCP)** is an open standard that enables AI models to:
- Connect securely to external data sources (files, databases, APIs)
- Use tools and execute actions
- Access real-time context from multiple sources through a standardized interface

Think of it as giving your AI model "eyes and hands" to interact with your data ecosystem.

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

from gen_ai_hub.proxy.native.openai import chat

def base_prompt(text):
    
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"{text}" 
                }
            ]
        }
    ]
    return messages

def write_prompt_messages(messages, model_name = "gpt-5"): # Models can be changed here
    """Send messages to the model and return the response."""
    kwargs = dict(model_name=model_name, messages=messages)
    response = chat.completions.create(**kwargs)
    return response.to_dict()["choices"][0]["message"]["content"]

def get_prompt_answer(text):
    """Get response from GPT 5"""
    messages = base_prompt(text)
    answer = write_prompt_messages(messages)
    return answer

get_prompt_answer("What are you?")

'I’m ChatGPT, an AI assistant created by OpenAI. I’m a large language model that can understand and generate text to help with questions, explanations, writing, coding, and more.\n\nA few things to know:\n- I don’t have feelings or consciousness.\n- I don’t access private data unless you share it in this chat.\n- My knowledge is based on training data and has a cutoff (October 2024), so I may not have the latest info.\n- I can make mistakes—please double-check important details.\n\nHow can I help you today?'

## Example 1: MCP Server for File System Access

Create a simple MCP server that gives your AI access to your project files:

In [25]:
import json
import os
from pathlib import Path

class SimpleMCPFileServer:
    """A simple MCP-inspired file server for demonstration"""
    
    def __init__(self, base_path="."):
        self.base_path = Path(base_path).resolve()
    
    def list_tools(self):
        """List available tools (MCP protocol method)"""
        return {
            "tools": [
                {
                    "name": "read_file",
                    "description": "Read contents of a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "path": {"type": "string", "description": "Relative path to file"}
                        },
                        "required": ["path"]
                    }
                },
                {
                    "name": "list_directory",
                    "description": "List files in a directory. Use '.' for the base directory.",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "path": {"type": "string", "description": "Relative path to directory (use '.' for root)"}
                        }
                    }
                },
                {
                    "name": "search_files",
                    "description": "Search for text in files",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "query": {"type": "string", "description": "Text to search for"},
                            "extension": {"type": "string", "description": "File extension filter (e.g., .py)"}
                        },
                        "required": ["query"]
                    }
                }
            ]
        }
    
    def call_tool(self, tool_name, arguments):
        """Execute a tool (MCP protocol method)"""
        if tool_name == "read_file":
            return self._read_file(arguments["path"])
        elif tool_name == "list_directory":
            return self._list_directory(arguments.get("path", "."))
        elif tool_name == "search_files":
            return self._search_files(arguments["query"], arguments.get("extension"))
        else:
            return {"error": f"Unknown tool: {tool_name}"}
    
    def _read_file(self, path):
        try:
            full_path = (self.base_path / path).resolve()
            if not str(full_path).startswith(str(self.base_path)):
                return {"error": "Access denied: path outside base directory"}
            with open(full_path, 'r') as f:
                return {"content": f.read()}
        except Exception as e:
            return {"error": str(e)}
    
    def _list_directory(self, path="."):
        try:
            # Handle empty or missing path
            if not path or path.strip() == "":
                path = "."
            
            full_path = (self.base_path / path).resolve()
            
            # Security check
            if not str(full_path).startswith(str(self.base_path)):
                return {"error": "Access denied: path outside base directory"}
            
            if not full_path.exists():
                return {"error": f"Directory not found: {path}"}
            
            if not full_path.is_dir():
                return {"error": f"Not a directory: {path}"}
            
            items = []
            for item in full_path.iterdir():
                items.append({
                    "name": item.name,
                    "type": "directory" if item.is_dir() else "file",
                    "size": item.stat().st_size if item.is_file() else None
                })
            return {"items": items, "path": str(path)}
        except Exception as e:
            return {"error": str(e)}
    
    def _search_files(self, query, extension=None):
        results = []
        try:
            for root, dirs, files in os.walk(self.base_path):
                for file in files:
                    if extension and not file.endswith(extension):
                        continue
                    file_path = Path(root) / file
                    try:
                        with open(file_path, 'r') as f:
                            content = f.read()
                            if query.lower() in content.lower():
                                rel_path = file_path.relative_to(self.base_path)
                                results.append(str(rel_path))
                    except:
                        pass
            return {"results": results}
        except Exception as e:
            return {"error": str(e)}

# Initialize the MCP server
mcp_server = SimpleMCPFileServer("../")

# List available tools
print("Available MCP Tools:")
print(json.dumps(mcp_server.list_tools(), indent=2))

Available MCP Tools:
{
  "tools": [
    {
      "name": "read_file",
      "description": "Read contents of a file",
      "inputSchema": {
        "type": "object",
        "properties": {
          "path": {
            "type": "string",
            "description": "Relative path to file"
          }
        },
        "required": [
          "path"
        ]
      }
    },
    {
      "name": "list_directory",
      "description": "List files in a directory. Use '.' for the base directory.",
      "inputSchema": {
        "type": "object",
        "properties": {
          "path": {
            "type": "string",
            "description": "Relative path to directory (use '.' for root)"
          }
        }
      }
    },
    {
      "name": "search_files",
      "description": "Search for text in files",
      "inputSchema": {
        "type": "object",
        "properties": {
          "query": {
            "type": "string",
            "description": "Text to search for"
         

## Example 2: Integrate MCP Tools with Your AI Core Setup

Now let's enhance your GenAI function to use MCP tools:

In [26]:
def enhanced_prompt_with_tools(user_query, use_tools=True):
    """Enhanced prompt that can use MCP tools"""
    
    # Start with the user's query
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant with access to file system tools. When users ask about files or data, use the available tools to provide accurate information."
        },
        {
            "role": "user",
            "content": user_query
        }
    ]
    
    if use_tools:
        # Add tool definitions to the context
        tools_info = mcp_server.list_tools()
        tools_description = "\n".join([
            f"- {tool['name']}: {tool['description']}" 
            for tool in tools_info['tools']
        ])
        
        messages[0]["content"] += f"\n\nAvailable tools:\n{tools_description}"
    
    # Get initial response
    response = write_prompt_messages(messages, model_name="gpt-5")
    
    return response

# Test the enhanced function
print("Testing enhanced AI with MCP context:")
print(enhanced_prompt_with_tools("What capabilities do you have with file access?"))

Testing enhanced AI with MCP context:
I can work with files in the current workspace using three read-only capabilities:

- List directory contents: I can show files and subfolders in a given path (default is the base directory “.”).
- Read files: I can open a file and return its text content.
- Search text in files: I can look for a given string across files and report matches with file names and line snippets.

What I can do with these:
- Navigate folders to find relevant files.
- Answer questions about file contents, summarize documents, extract data, or compare files by reading them.
- Search for keywords (e.g., TODOs, errors, function names) across the codebase or documents.
- Parse common text formats (code, logs, JSON, CSV, Markdown) and help analyze them.

Limitations:
- Read-only: I can’t create, edit, or delete files.
- Scope: Access is limited to the workspace provided in this session; I can’t reach your local machine, external drives, or the internet.
- Execution: I can’t r

## Example 3: Agent Loop with Tool Execution

Create an agent that can actually execute MCP tools based on AI decisions:

In [27]:
import re

def mcp_agent(user_query, max_iterations=5):
    """
    An agent that can use MCP tools to answer queries.
    This is a simplified version - production would use function calling APIs.
    """
    
    tools = mcp_server.list_tools()["tools"]
    tools_description = "\n".join([
        f"- {tool['name']}: {tool['description']}" 
        for tool in tools
    ])
    
    conversation = [
        {
            "role": "system",
            "content": f"""You are an AI assistant with access to tools. When you need information, 
request tool usage in this format: TOOL_CALL: tool_name(arguments_as_json)

Available tools:
{tools_description}

Example: TOOL_CALL: read_file({{"path": "README.md"}})"""
        },
        {
            "role": "user",
            "content": user_query
        }
    ]
    
    for iteration in range(max_iterations):
        # Get AI response
        response = write_prompt_messages(conversation, model_name="gpt-4o")
        
        print(f"\n=== Iteration {iteration + 1} ===")
        print(f"AI: {response[:200]}..." if len(response) > 200 else f"AI: {response}")
        
        # Check if AI wants to call a tool
        tool_match = re.search(r'TOOL_CALL:\s*(\w+)\((.*?)\)', response, re.DOTALL)
        
        if tool_match:
            tool_name = tool_match.group(1)
            args_str = tool_match.group(2)
            
            try:
                # Parse arguments
                arguments = json.loads(args_str)
                print(f"Calling tool: {tool_name} with {arguments}")
                
                # Execute tool
                result = mcp_server.call_tool(tool_name, arguments)
                
                # Add tool result to conversation
                conversation.append({
                    "role": "assistant",
                    "content": response
                })
                conversation.append({
                    "role": "user",
                    "content": f"Tool result: {json.dumps(result)}"
                })
                
            except Exception as e:
                print(f"Error executing tool: {e}")
                break
        else:
            # No tool call, we're done
            return response
    
    return response

# Test the agent
print("="*60)
print("MCP AGENT DEMO")
print("="*60)
result = mcp_agent("List the files and directories in the root directory. Use list_directory with path '.'")
print(f"\n✅ Final result: {result}")

MCP AGENT DEMO

=== Iteration 1 ===
AI: TOOL_CALL: list_directory({"path": "."})
Calling tool: list_directory with {'path': '.'}

=== Iteration 2 ===
AI: The root directory contains the following files and directories:

Files:
1. `gdelt.ipynb` (198,319 bytes)
2. `.DS_Store` (8,196 bytes)
3. `requirements.txt` (2,819 bytes)
4. `runtime.txt` (12 bytes)
5...

✅ Final result: The root directory contains the following files and directories:

Files:
1. `gdelt.ipynb` (198,319 bytes)
2. `.DS_Store` (8,196 bytes)
3. `requirements.txt` (2,819 bytes)
4. `runtime.txt` (12 bytes)
5. `geospatial.ipynb` (17,734 bytes)
6. `README.md` (404 bytes)
7. `.gitignore` (75 bytes)
8. `.env` (1,170 bytes)

Directories:
1. `ukraine`
2. `netherlands`
3. `rpt-1`
4. `explosives`
5. `germany`
6. `mcp`
7. `.venv`
8. `msc`
9. `.git`


## Next Steps & Best Practices

### 1. **Production MCP Servers**
Instead of building custom servers, use official MCP servers:
- `@modelcontextprotocol/server-filesystem` - File access
- `@modelcontextprotocol/server-github` - GitHub integration  
- `@modelcontextprotocol/server-postgres` - Database access
- Custom servers for your specific data sources

### 2. **Use Function Calling APIs**
Modern LLM APIs (OpenAI, Anthropic) support native function calling:
```python
# With OpenAI function calling:
tools = [{
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Read a file",
        "parameters": {...}
    }
}]
response = client.chat.completions.create(
    model="gpt-4",
    messages=messages,
    tools=tools
)
```

### 3. **Security Considerations**
- Validate all file paths to prevent directory traversal
- Implement access control and audit logging
- Rate limit tool executions
- Sanitize tool outputs before sending to LLM

### 4. **Integration with SAP AI Core**
- Wrap MCP servers as SAP AI Core serving endpoints
- Use SAP AI Core's authentication for secure access
- Store MCP configurations in SAP BTP

### 5. **Advanced Patterns**
- **Streaming**: Stream tool results for better UX
- **Caching**: Cache tool results to reduce latency
- **Parallel Tool Calls**: Execute independent tools concurrently
- **Resource Management**: Connect MCP to databases, APIs, cloud services

### Resources:
- MCP Specification: https://modelcontextprotocol.io
- MCP Python SDK: https://github.com/modelcontextprotocol/python-sdk
- MCP Servers: https://github.com/modelcontextprotocol/servers

---

## Bonus: Native OpenAI Function Calling (No MCP!)

For comparison, here's how to achieve the same thing using OpenAI's built-in function calling API - simpler and no custom protocol needed:

In [28]:
# Native OpenAI Function Calling - No MCP Required!
from gen_ai_hub.proxy.native.openai import chat
import json
import os
from pathlib import Path

# Define your functions
def list_directory_func(path="."):
    """List files and directories"""
    base_path = Path("../").resolve()
    full_path = (base_path / path).resolve()
    
    if not str(full_path).startswith(str(base_path)):
        return {"error": "Access denied"}
    
    items = []
    for item in full_path.iterdir():
        items.append({
            "name": item.name,
            "type": "directory" if item.is_dir() else "file",
            "size": item.stat().st_size if item.is_file() else None
        })
    return {"items": items, "path": str(path)}

def read_file_func(path):
    """Read a file's contents"""
    base_path = Path("../").resolve()
    full_path = (base_path / path).resolve()
    
    if not str(full_path).startswith(str(base_path)):
        return {"error": "Access denied"}
    
    try:
        with open(full_path, 'r') as f:
            return {"content": f.read()[:1000]}  # First 1000 chars
    except Exception as e:
        return {"error": str(e)}

# Define tools in OpenAI format
tools = [
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": "List files and directories at a path",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative path (use '.' for root)"
                    }
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read contents of a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative path to file"
                    }
                },
                "required": ["path"]
            }
        }
    }
]

# Function dispatcher
available_functions = {
    "list_directory": list_directory_func,
    "read_file": read_file_func
}

# Run the agent
messages = [{
    "role": "user",
    "content": "List the files in the root directory and describe the structure"
}]

print("🚀 OpenAI Native Function Calling Demo\n")
print("="*60)

for iteration in range(5):
    # Call the model
    response = chat.completions.create(
        model_name="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_dict = response.to_dict()
    response_message = response_dict["choices"][0]["message"]
    
    # Check if the model wants to call a function
    tool_calls = response_message.get("tool_calls")
    
    if not tool_calls:
        # No more function calls, we have the final answer
        final_answer = response_message.get("content")
        print(f"\n✅ Final Answer:\n{final_answer}")
        break
    
    # Add assistant's response to messages
    messages.append(response_message)
    
    # Execute each function call
    for tool_call in tool_calls:
        function_name = tool_call["function"]["name"]
        function_args = json.loads(tool_call["function"]["arguments"])
        
        print(f"\n🔧 Calling: {function_name}({function_args})")
        
        # Call the function
        function_response = available_functions[function_name](**function_args)
        
        # Add function response to messages
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "name": function_name,
            "content": json.dumps(function_response)
        })

print("\n" + "="*60)
print("✨ This is cleaner than MCP for single-vendor scenarios!")

🚀 OpenAI Native Function Calling Demo


🔧 Calling: list_directory({'path': '.'})

✅ Final Answer:
The root directory contains the following structure:

### Files:
1. **gdelt.ipynb**: A Jupyter Notebook file, 198319 bytes in size.
2. **.DS_Store**: A system file typically generated by macOS, 8196 bytes in size.
3. **requirements.txt**: A text file, likely containing a list of dependencies for a Python project, 2819 bytes in size.
4. **runtime.txt**: A text file, possibly specifying the runtime environment, e.g., Python version, 12 bytes in size.
5. **geospatial.ipynb**: Another Jupyter Notebook file related to geospatial analyses, 17734 bytes in size.
6. **README.md**: A README file, typically used for documentation, 404 bytes in size.
7. **.gitignore**: A configuration file for Git, used to specify files and directories to be ignored by version control, 75 bytes in size.
8. **.env**: A file likely containing environment variables, 1170 bytes in size.

### Directories:
1. **ukraine**: A